In [0]:
from pyspark.sql.functions import col

print("1. Reading Raw 2018-2024 Parquet from Volume...")
file_path = "/Volumes/workspace/aviation_project/raw_data/flights_2018_2024.parquet"

# Parquet natively stores its own schema, so we just read it directly.
raw_flights = spark.read.parquet(file_path)
print(f"Read {raw_flights.count()} records from {file_path}.")

In [0]:
print("2. Selecting specific columns to prevent wide-table bloat...")
bronze_flights = raw_flights.select(
    col("Year"),
    col("FlightDate"),
    col("Operating_Airline ").alias("Operating_Airline"),
    col("Flight_Number_Operating_Airline"),
    col("Origin"),
    col("Dest"),
    col("CRSDepTime"),
    col("DepDelay"),
    col("Cancelled"),
    col("Diverted"),
    col("Distance")
)

In [0]:
print("3. Writing to Bronze Delta Table...")
bronze_flights.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("aviation_project.bronze_historical_flights")

In [0]:
final_count = spark.table("aviation_project.bronze_historical_flights").count()
print(f"4. SUCCESS! Ingested {final_count} historical flight records into Bronze.")